In [ ]:
import pandas as pd
import numpy as np

# Load your raw file
# Note: encoding='utf-8' is standard, but some gov files use 'ISO-8859-1'
try:
    df = pd.read_csv("../data/raw/dins_raw.csv", encoding='utf-8')
except UnicodeDecodeError:
    df = pd.read_csv("../data/raw/dins_raw.csv", encoding='ISO-8859-1')

print(f"Original Rows: {len(df)}")
print("Columns found:", df.columns.tolist())

ModuleNotFoundError: No module named 'pandas'

In [ ]:
# 1. Define the columns we actually need based on your screenshots
keep_cols = [
    'AIN',               # Unique ID
    'SitusFullAddress',  # Address (for debugging)
    'CENTER_LAT',        # Coordinates
    'CENTER_LON', 
    '* Damage',          # The Target Label (Note the space in name)
    'Structure Category' # To filter for "Single Family" etc.
]

# 2. Select and Rename for sanity
df_clean = df[keep_cols].copy()
df_clean.columns = ['id', 'address', 'lat', 'lon', 'damage_str', 'structure_type']

# 3. Drop rows with missing essential data
df_clean = df_clean.dropna(subset=['lat', 'lon', 'damage_str'])
print(f"Rows after dropping missing coordinates: {len(df_clean)}")

In [ ]:
# Check what types exist
print("Structure Types:", df_clean['structure_type'].unique())

# FILTER LOGIC: Adjust this list based on the print output above!
# Usually we want 'Single Family...', 'Multi-Family...', 'Mobile Home'
valid_types = [
    'Single Residence', 
    'Multiple Residence', 
    'Single Family Residence', 
    'Residential'
]

In [ ]:
# Check unique damage values
print("Damage Categories:", df_clean['damage_str'].unique())

def map_damage(val):
    val = str(val).lower()
    # BURNED (1)
    if 'destroyed' in val or 'major' in val:
        return 1
    # SURVIVED (0)
    # Includes 'Minor', 'Affected', 'No Damage'
    elif 'minor' in val or 'affected' in val or 'no damage' in val:
        return 0
    else:
        return np.nan # Drop ambiguous cases

df_clean['target'] = df_clean['damage_str'].apply(map_damage)
df_clean = df_clean.dropna(subset=['target'])

print("Target Distribution:")
print(df_clean['target'].value_counts())

In [ ]:
# Separate
burned = df_clean[df_clean['target'] == 1]
survived = df_clean[df_clean['target'] == 0]

# Balance (Optional but recommended for training)
# We take all burned, and an equal number of survived
n_samples = len(burned)
if len(survived) > n_samples:
    survived = survived.sample(n=n_samples, random_state=42)

final_df = pd.concat([burned, survived]).sample(frac=1).reset_index(drop=True)

print(f"Final Dataset: {len(final_df)} homes ({len(burned)} burned, {len(survived)} survived)")

# Save
final_df.to_csv("../data/processed/clean_homes.csv", index=False)
print("Saved to data/processed/clean_homes.csv")